# CV-QKD Key-Rate Analysis

Generated from the public notebook builder for reproducible analysis.

Notebook 04 visualised the GG02 protocol in phase space and ran the Alice-Bob mutual information. This notebook closes the loop: the Holevo bound is computed from the symplectic eigenvalues of the joint covariance matrix, and the per-symbol secure-key rate is

$$K = \max\bigl(0,\ \beta\,I(A:B) - \chi(B:E)\bigr)$$

with $\beta$ the reverse-reconciliation efficiency and $\chi$ the asymptotic collective-attack Holevo bound from Laudenbach et al. (2018), Sections 6&ndash;7.

**Convention contract.** Vacuum variance = 1, $\hbar = 2$, $\nu \ge 1$ for physical states.

**Untrusted-detector model.** Total $\eta = \eta_\mathrm{ch} \times \eta_\mathrm{det}$ is attributed to the channel available to Eve.

**Range pre-claim caveat .** The maximum secure distance reported below is computed under the *stated* parameter set; it is not a universal CV-QKD-vs-BB84 ranking.

## 1. Bootstrap and imports

In [1]:
from pathlib import Path
import sys


def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise RuntimeError('Could not find project root')


PROJECT_ROOT = find_project_root()
FIG_DIR = PROJECT_ROOT / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')


Project root: C:\Users\COWLAR\projects\qkd-protocol-simulator


In [2]:
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
from scipy.optimize import brentq

from src.cvqkd import (
    cvqkd_holevo_bound_homodyne,
    cvqkd_key_rate,
    cvqkd_mutual_info_homodyne,
)
from src.channel import bb84_key_rate, fiber_transmittance
from src.plotting import semilogy_positive

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 150})
L = np.linspace(0.0, 300.0, 1001)
